## 0. Setup (unchanged from your working notebook)

In [1]:
%pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.5/304.5 kB 5.0 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
import re
import json
import time
import random
import requests
import pandas as pd
import numpy as np
from openai import OpenAI
from tqdm import tqdm
from typing import List, Dict, Optional

# Load credentials from Kaggle Secrets when running on Kaggle;
# otherwise load them from environment variables.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    NVIDIA_API_KEY = secrets.get_secret("NVIDIA_API_KEY")
    FIRECRAWL_API_KEY = secrets.get_secret("FIRECRAWL_API_KEY")
except Exception:
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
    FIRECRAWL_API_KEY = os.getenv("FIRECRAWL_API_KEY")

if not NVIDIA_API_KEY or not FIRECRAWL_API_KEY:
    raise EnvironmentError(
        "Set NVIDIA_API_KEY and FIRECRAWL_API_KEY as environment variables "
        "or add them to Kaggle Secrets before running this notebook."
    )

print("Wikipedia live API — no key required")

# NVIDIA NIM client (OpenAI-compatible API)
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

# Firecrawl client
from firecrawl import FirecrawlApp
firecrawl_client = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

# Model configuration
MODELS = {
    "generation": "nvidia/nemotron-3-super-120b-a12b",
    "extraction": "nvidia/nemotron-3-ultra-550b-a55b",
    "explanation": "nvidia/nemotron-3-ultra-550b-a55b"
}

FALLBACK_MODELS = {
    "generation": ["nvidia/nemotron-3-super-120b-a12b", "nvidia/nemotron-3-ultra-550b-a55b"],
    "extraction": ["nvidia/nemotron-3-ultra-550b-a55b", "nvidia/nemotron-3-super-120b-a12b"],
    "explanation": ["nvidia/nemotron-3-ultra-550b-a55b", "nvidia/nemotron-3-super-120b-a55b"]
}

NLI_THRESHOLD = 0.5
SEARCH_RESULT_COUNT = 3
WIKI_RESULT_COUNT = 2
MAX_CONTENT_CHARS = 800

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Configuration loaded")
print(f"Generation model : {MODELS['generation']}")
print(f"Extraction model : {MODELS['extraction']}")
print("Tier 1           : Wikipedia live API")
print("Tier 2           : Firecrawl live web search")
print("NLI              : cross-encoder/nli-deberta-v3-small (local)")
print(f"NLI threshold    : {NLI_THRESHOLD}")


✅ Keys loaded from Kaggle Secrets
✅ Wikipedia live API — no key required
✅ NVIDIA NIM client + Firecrawl client ready

✅ Configuration loaded
   Step 2 — Generation : nvidia/nemotron-3-super-120b-a12b
   Step 3 — Extraction : nvidia/nemotron-3-ultra-550b-a55b
   Step 4 — Tier 1     : Wikipedia live API (free, no key)
   Step 4 — Tier 2     : Firecrawl live web search (escalation only)
   Step 5 — NLI        : cross-encoder/nli-deberta-v3-small (local)
   Step 6 — Explanation: nvidia/nemotron-3-ultra-550b-a55b
   NLI Threshold       : 0.5


In [3]:
def call_nvidia_nim(
    model      : str,
    messages   : List[Dict],
    temperature: float = 0,
    max_tokens : int   = 1024,
    retries    : int   = 3
) -> Optional[str]:
    """
    Call NVIDIA NIM API using OpenAI-compatible SDK.
    Returns response text or None if all retries fail.
    """
    
    for attempt in range(retries):
        try:
            response = nvidia_client.chat.completions.create(
                model=model, messages=messages, temperature=temperature,
                max_tokens=max_tokens, timeout=60,
                extra_body={"chat_template_kwargs": {"thinking": False}}  # NEW: suppress visible reasoning
            )
            content = response.choices[0].message.content
            if content is None:
                print(f"   ⚠️  {model} returned empty content (attempt {attempt+1})")
                if attempt == retries - 1:
                    return None
                time.sleep(3)
                continue
            return content.strip()
        
        except Exception as e:
            err_str = str(e).lower()

            if "404" in err_str or "410" in err_str or "not found" in err_str or "gone" in err_str:
                # Dead/unavailable/not-enabled-for-account endpoint —
                # retrying never helps, fail fast to next fallback
                code = "410" if "410" in err_str else "404"
                print(f"   ❌ {model} — endpoint unavailable ({code}), skipping to next fallback")
                return None

            elif "429" in err_str or "rate" in err_str:
                wait = 20 * (attempt + 1)
                print(f"   ⚠️  Rate limit hit on {model}. Waiting {wait}s...")
                time.sleep(wait)

            elif "503" in err_str or "resourceexhausted" in err_str:
                wait = 15 * (attempt + 1)
                print(f"   ⚠️  {model} overloaded (503). Waiting {wait}s...")
                time.sleep(wait)

            elif "timeout" in err_str or "timed out" in err_str:
                print(f"   ⚠️  {model} timed out (attempt {attempt+1})")
                if attempt == retries - 1:
                    return None
                time.sleep(5)

            else:
                print(f"   ❌ Error on {model} (attempt {attempt+1}): {str(e)[:120]}")
                time.sleep(5)

    return None


def call_with_fallback(
    step       : str,
    messages   : List[Dict],
    temperature: float = 0,
    max_tokens : int   = 1024
) -> Optional[str]:
    """Try primary model for the step, then fallbacks in order."""
    for model in FALLBACK_MODELS[step]:
        result = call_nvidia_nim(model, messages, temperature, max_tokens)
        if result:
            return result
        print(f"   ↪ Trying next fallback for step '{step}'...")

    print(f"   ❌ All fallbacks exhausted for step '{step}'")
    return None


print("✅ NVIDIA NIM caller functions ready (None-content guard, 404/410 fast-fail, 503/429 backoff)")

✅ NVIDIA NIM caller functions ready (None-content guard, 404/410 fast-fail, 503/429 backoff)


In [4]:
# TEST NVIDIA NIM CONNECTION
print("🧪 Testing NVIDIA NIM connection...")
test_response = call_nvidia_nim(
    model      = MODELS["generation"],
    messages   = [{"role": "user", "content": "Say hello in exactly one sentence."}],
    temperature= 0,
    max_tokens = 30
)
if test_response:
    print(f"✅ NVIDIA NIM connected!")
    print(f"   Model    : {MODELS['generation']}")
    print(f"   Response : {test_response}")
else:
    print("❌ Connection failed — check your NVIDIA_API_KEY")

🧪 Testing NVIDIA NIM connection...
✅ NVIDIA NIM connected!
   Model    : nvidia/nemotron-3-super-120b-a12b
   Response : Hello.


## 1. v3 configuration

In [7]:
import hashlib
from typing import List, Dict, Optional, Tuple

# ─────────────────────────────────────────────────────────────
# v3 CONFIGURATION  (the ONLY cell you normally need to edit)
# ─────────────────────────────────────────────────────────────

# Wikimedia asks every client to identify itself. Put a real email here.
WIKI_CONTACT = "YOUR_EMAIL@example.com"  # Replace with your contact email

# Experiment size / reproducibility
N_PAIRS   = 50     # questions sampled; each gives 1 truthful + 1 hallucinated sample
EVAL_SEED = 11     # seed 7 was used for the first (v3.0) run that guided these fixes.
                   # Keep 11 (a fresh sample) for the final numbers; never re-tune on the sample you report.

# Retrieval
WIKI_MAX_PAGES        = 3      # Wikipedia pages fetched per claim (full text, not just intro)
FIRECRAWL_MAX_CHARS   = 20000  # keep up to this much text per web page (old code kept 800)
CHUNK_CHARS           = 500    # evidence is split into chunks of about this size
MIN_CHUNK_CHARS       = 80     # chunks shorter than this (headings, list items) are ignored when ranking
TOP_K_CHUNKS          = 5      # best-matching chunks passed to the verifier
EVIDENCE_CHAR_BUDGET  = 2500   # hard cap on evidence text given to the judge
WIKI_SLEEP            = 0.3    # polite pause between Wikipedia requests (seconds)
FIRECRAWL_SLEEP       = 1.0    # pause after each Firecrawl call (seconds)

# Judge (HalluMeasure-style chain-of-thought claim classifier)
JUDGE_MAX_TOKENS  = 700
# Tier-1 (Wikipedia) is accepted when the judge is CONCLUSIVE about the claim; otherwise escalate to Firecrawl
CONCLUSIVE_LABELS = ("SUPPORTED", "CONTRADICTED")

# Extra model "steps" for call_with_fallback(); they reuse the models already
# confirmed working on your account (see MODELS / FALLBACK_MODELS above).
FALLBACK_MODELS["judge"]    = list(FALLBACK_MODELS["explanation"])   # strongest model first
FALLBACK_MODELS["claimgen"] = list(FALLBACK_MODELS["generation"])    # cheaper model first

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# Simple health counters so the results section can prove the Wikipedia fix worked
WIKI_STATS      = {"requests": 0, "retried": 0, "errors": 0, "last_error": ""}
FIRECRAWL_STATS = {"searches": 0, "retried": 0, "errors": 0, "last_error": ""}

print("✅ v3 config loaded")
print(f"   Pairs (samples)  : {N_PAIRS} ({2 * N_PAIRS})   seed={EVAL_SEED}")
print(f"   Judge models     : {FALLBACK_MODELS['judge']}")
print(f"   Claim-gen models : {FALLBACK_MODELS['claimgen']}")
print(f"   Output directory : {WORK_DIR}")
if WIKI_CONTACT.startswith("your-email"):
    print("⚠️  Set WIKI_CONTACT to your real email (Wikipedia rate-limits anonymous clients harder).")


✅ v3 config loaded
   Pairs (samples)  : 50 (100)   seed=11
   Judge models     : ['nvidia/nemotron-3-ultra-550b-a55b', 'nvidia/nemotron-3-super-120b-a12b']
   Claim-gen models : ['nvidia/nemotron-3-super-120b-a12b', 'nvidia/nemotron-3-ultra-550b-a55b']
   Output directory : /kaggle/working


## 2. NLI model (baseline verifier — unchanged)

In [8]:
import torch
from transformers import pipeline

device      = 0 if torch.cuda.is_available() else -1
device_name = "GPU (CUDA)" if torch.cuda.is_available() else "CPU"
print(f"🖥️  NLI running on: {device_name}")
print("📥 Loading cross-encoder/nli-deberta-v3-small (~180MB)...")

nli_model = pipeline(
    "text-classification",
    model ="cross-encoder/nli-deberta-v3-small",
    device= device,
    top_k = None   # FIX: return ALL label scores, not just top-1
)

print("✅ NLI model loaded")


def clean_evidence_text(text: str) -> str:
    """FIX: strip markdown links/nav boilerplate that was diluting entailment scores."""
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)   # [text](url) -> text
    text = re.sub(r'!\[[^\]]*\]\([^\)]+\)', '', text)        # remove images
    boilerplate = [
        r'Jump to content', r'Skip to main content',
        r'From Wikipedia, the free encyclopedia',
        r'This article contains.*?characters\.?',
    ]
    for pat in boilerplate:
        text = re.sub(pat, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_entailment_score(claim: str, evidence_text: str) -> float:
    """
    Check if evidence_text entails (supports) the claim.
    Returns score 0.0 (contradicted/unsupported) to 1.0 (fully supported)
    """
    evidence_clean     = clean_evidence_text(evidence_text)
    evidence_truncated = evidence_clean[:700]
    try:
        results = nli_model(
            {"text": evidence_truncated, "text_pair": claim},
            truncation=True,
            max_length=512
        )
        for r in results:
            if r["label"].lower() == "entailment":
                return float(r["score"])
        return 0.0
    except Exception as e:
        print(f"   ❌ NLI error: {e}")
        return 0.0


🖥️  NLI running on: GPU (CUDA)
📥 Loading cross-encoder/nli-deberta-v3-small (~180MB)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

✅ NLI model loaded


In [9]:
# Sanity check — confirm label mapping and scoring behave as expected
print(nli_model.model.config.id2label)
print()
test_out = get_entailment_score(
    claim="King Sejong ruled from 1418 to 1450.",
    evidence_text="King of Joseon from 1418 to 1450"
)
print(f"Entailment score for a clearly-supported claim: {test_out:.3f} (should be high, ~0.8+)")

{0: 'contradiction', 1: 'entailment', 2: 'neutral'}

Entailment score for a clearly-supported claim: 0.799 (should be high, ~0.8+)


## 3. Retrieval: Wikipedia (Tier 1, fixed) → Firecrawl (Tier 2), chunk & rank

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

WIKI_API     = "https://en.wikipedia.org/w/api.php"
WIKI_HEADERS = {"User-Agent": f"WebHaluCheck-StudentProject/2.0 (educational use; contact: {WIKI_CONTACT})"}

BLOCKED_PHRASES = [
    "unsupported browser", "enable javascript", "access denied",
    "captcha", "are you a robot", "please verify you are human"
]
# Pages that are almost never useful factual evidence
DENY_DOMAINS = ("open.spotify.com", "facebook.com", "instagram.com", "tiktok.com",
                "youtube.com", "youtu.be", "twitter.com", "x.com", "pinterest.com")


def _extract_url(r):
    url = getattr(r, "url", "") or ""
    if not url:
        meta = getattr(r, "metadata", None)
        if meta is not None:
            url = getattr(meta, "url", "") or getattr(meta, "source_url", "") or ""
    return url


# ─────────────────────────────────────────────────────────────
# WIKIPEDIA (Tier 1) — fixed: checks HTTP status, backs off on 429/5xx
# ─────────────────────────────────────────────────────────────
def _wiki_get(params: Dict, retries: int = 4) -> Optional[Dict]:
    """GET the Wikipedia API. Returns parsed JSON, or None after retries.
    The old code called .json() without checking the status, which is what produced
    'Expecting value: line 1 column 1 (char 0)' whenever Wikipedia returned an HTML/empty error page."""
    for attempt in range(retries):
        WIKI_STATS["requests"] += 1
        try:
            resp = requests.get(WIKI_API, params=params, headers=WIKI_HEADERS, timeout=15)
        except requests.RequestException as e:
            WIKI_STATS["last_error"] = f"network: {str(e)[:80]}"
            WIKI_STATS["retried"] += 1
            time.sleep(1.5 * (attempt + 1))
            continue

        if resp.status_code == 200:
            try:
                return resp.json()
            except ValueError:
                WIKI_STATS["last_error"] = "HTTP 200 but body was not JSON"
                WIKI_STATS["retried"] += 1
                time.sleep(1.5 * (attempt + 1))
                continue

        WIKI_STATS["last_error"] = f"HTTP {resp.status_code}"
        if resp.status_code in (429, 500, 502, 503, 504):
            WIKI_STATS["retried"] += 1
            wait = 2.0 * (attempt + 1)
            try:
                wait = max(wait, float(resp.headers.get("Retry-After", 0)))
            except (TypeError, ValueError):
                pass
            time.sleep(min(wait, 30))
            continue
        break   # 403 / 400 etc. — retrying will not help

    WIKI_STATS["errors"] += 1
    return None


def wiki_search_titles(query: str, limit: int) -> List[str]:
    data = _wiki_get({"action": "query", "list": "search", "srsearch": query[:300],
                      "srlimit": limit, "format": "json"})
    if not data:
        return []
    return [r["title"] for r in data.get("query", {}).get("search", []) if "title" in r]


def wiki_fetch_text(title: str) -> str:
    """Full plain-text article (not only the intro — HaluEval facts live in the body)."""
    data = _wiki_get({"action": "query", "prop": "extracts", "explaintext": 1, "redirects": 1,
                      "titles": title, "format": "json"})
    if not data:
        return ""
    for page in data.get("query", {}).get("pages", {}).values():
        text = (page.get("extract") or "").strip()
        if text:
            return text[:60000]
    return ""


_Q_STOP = {"In", "Which", "What", "Who", "Whom", "Whose", "Where", "When", "How", "Why", "The", "A", "An",
           "Is", "Are", "Was", "Were", "Did", "Does", "Do", "Of", "On", "At", "For", "And", "Or", "By", "From",
           "To", "With", "That", "This", "These", "Those", "His", "Her", "Its", "Their", "He", "She", "It",
           "They", "Answer", "Question", "Yes", "No"}


def entity_query(text: str) -> str:
    """Wikipedia search requires ALL words to match, so long sentences often return nothing.
    This keeps only capitalised words / years (the entities), e.g.
    'Which opera was performed first The Midsummer Marriage or Die Feen?' -> 'Midsummer Marriage Die Feen'."""
    toks = re.findall(r"[A-Z][\w'’\-]*|\d{3,4}", text or "")
    toks = [t for t in toks if t not in _Q_STOP]
    return " ".join(list(dict.fromkeys(toks))[:8])


def wikipedia_docs(statement: str, question: Optional[str] = None) -> List[Dict]:
    """Search with the claim AND the question (multi-hop facts often sit on the question entity's page).
    If both natural-language queries return nothing, retry with entity-only queries."""
    titles: List[str] = []

    def _add(q: str):
        for t in wiki_search_titles(q, WIKI_RESULT_COUNT):
            if t not in titles:
                titles.append(t)
        time.sleep(WIKI_SLEEP)

    for q in dict.fromkeys([x for x in (statement, question) if x]):
        _add(q)
    if not titles:
        for q in dict.fromkeys([entity_query(question or ""), entity_query(statement)]):
            if q:
                _add(q)
    docs = []
    for t in titles[:WIKI_MAX_PAGES]:
        text = wiki_fetch_text(t)
        if len(text) > 100:
            docs.append({"title": t, "url": f"https://en.wikipedia.org/wiki/{t.replace(' ', '_')}",
                         "text": text, "source": "wikipedia"})
        time.sleep(WIKI_SLEEP)
    return docs


# ─────────────────────────────────────────────────────────────
# FIRECRAWL (Tier 2) — fixed: keeps the whole page, not the first 800 characters
# ─────────────────────────────────────────────────────────────
def firecrawl_docs(query: str, limit: Optional[int] = None) -> List[Dict]:
    limit = limit or SEARCH_RESULT_COUNT
    response = None
    for attempt in range(2):
        FIRECRAWL_STATS["searches"] += 1
        try:
            response = firecrawl_client.search(query=query, limit=limit,
                                               scrape_options={"formats": ["markdown"]})
            break
        except Exception as e:
            FIRECRAWL_STATS["retried"] += 1
            FIRECRAWL_STATS["last_error"] = str(e)[:120]
            time.sleep(3)
    if response is None:
        FIRECRAWL_STATS["errors"] += 1
        return []

    docs = []
    for r in (getattr(response, "web", None) or []):
        url = _extract_url(r)
        if any(d in url.lower() for d in DENY_DOMAINS):
            continue
        raw = (getattr(r, "markdown", "") or getattr(r, "description", "")
               or getattr(r, "content", "") or "")
        text = clean_evidence_text(raw)[:FIRECRAWL_MAX_CHARS]
        if len(text) < 100 or any(p in text[:600].lower() for p in BLOCKED_PHRASES):
            continue
        docs.append({"title": getattr(r, "title", "") or url, "url": url,
                     "text": text, "source": "firecrawl"})
    return docs


# ─────────────────────────────────────────────────────────────
# CHUNK + RANK — replaces "first 800 characters"
# ─────────────────────────────────────────────────────────────
def chunk_text(text: str, max_chars: int = None) -> List[str]:
    max_chars = max_chars or CHUNK_CHARS
    chunks: List[str] = []
    for para in re.split(r"\n+", text):
        para = para.strip()
        if not para or re.match(r"^=+\s*.*?\s*=+$", para):      # skip empty lines / wiki headings
            continue
        cur = ""
        for s in re.split(r"(?<=[.!?])\s+", para):
            if not s:
                continue
            if len(s) > max_chars:                              # tables / run-on text: hard split
                if cur:
                    chunks.append(cur)
                    cur = ""
                chunks.extend(s[i:i + max_chars] for i in range(0, len(s), max_chars))
                continue
            if cur and len(cur) + len(s) + 1 > max_chars:
                chunks.append(cur)
                cur = s
            else:
                cur = (cur + " " + s).strip()
        if cur:
            chunks.append(cur)
    return [c for c in chunks if len(c) >= 25]


def select_evidence(query: str, docs: List[Dict], top_k: int = None, char_budget: int = None) -> List[Dict]:
    """Split all docs into chunks, rank by TF-IDF cosine similarity to `query`, keep the best few."""
    top_k       = top_k or TOP_K_CHUNKS
    char_budget = char_budget or EVIDENCE_CHAR_BUDGET

    items, seen = [], set()
    for d in docs:
        for c in chunk_text(d["text"]):
            key = c.lower()
            if key in seen:
                continue
            seen.add(key)
            items.append({"text": c, "title": d["title"], "url": d["url"], "source": d["source"]})
    if not items:
        return []
    # Short heading-like chunks ("List of accidents at JFK", "Alex Karras (1935-2012)") get inflated TF-IDF
    # cosine scores and used to outrank the real sentences. Ignore them unless nothing else exists.
    long_items = [it for it in items if len(it["text"]) >= MIN_CHUNK_CHARS]
    if long_items:
        items = long_items

    texts = [it["text"] for it in items]
    try:
        vec  = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
        M    = vec.fit_transform(texts + [query])
        sims = (M[:-1] @ M[-1].T).toarray().ravel()
    except ValueError:                       # empty vocabulary (e.g. only stop-words)
        sims = np.zeros(len(items))

    selected, used = [], 0
    for idx in np.argsort(-sims, kind="stable"):
        if len(selected) >= top_k:
            break
        it = items[int(idx)]
        if selected and used + len(it["text"]) > char_budget:
            break
        selected.append({**it, "sim": round(float(sims[idx]), 4)})
        used += len(it["text"])
    return selected


print("✅ Retrieval layer ready (Wikipedia status-checked + full text, Firecrawl full pages, chunk & rank)")


✅ Retrieval layer ready (Wikipedia status-checked + full text, Firecrawl full pages, chunk & rank)


In [11]:
# SMOKE TEST — proves Wikipedia now works and that the right sentence is retrieved
# (the old notebook returned only the intro and often failed with "Expecting value: line 1 column 1")
_t_claim = "King Sejong ruled Joseon from 1418 to 1450."
_t_docs  = wikipedia_docs(_t_claim, "Who was King Sejong?")
print(f"📖 Wikipedia pages fetched : {len(_t_docs)}  {[d['title'] for d in _t_docs]}")
print(f"   API requests / retried / failed : {WIKI_STATS['requests']} / {WIKI_STATS['retried']} / {WIKI_STATS['errors']}" + (f"   (last transient error: {WIKI_STATS['last_error']})" if WIKI_STATS['last_error'] else ""))

_t_chunks = select_evidence(_t_claim + " Who was King Sejong?", _t_docs)
print(f"   Evidence chunks kept    : {len(_t_chunks)}")
for _c in _t_chunks[:3]:
    print(f"   • sim={_c['sim']:.3f} | {_c['title']}: {_c['text'][:160]}...")

if not _t_docs:
    print("\n❌ No Wikipedia pages came back. Check WIKI_STATS['last_error'] above:")
    print("   HTTP 429/403 → set WIKI_CONTACT to a real email, wait a minute, re-run this cell.")
else:
    print("\n✅ Wikipedia retrieval OK")


📖 Wikipedia pages fetched : 3  ['Sejong the Great', 'Sejo of Joseon', 'King Sejong Station']
   API requests / retried / failed : 5 / 0 / 0
   Evidence chunks kept    : 5
   • sim=0.201 | King Sejong Station: The King Sejong Station (Korean: 세종과학기지) is a research station for the Korea Antarctic Research Program that is named after King Sejong the Great of Joseon (139...
   • sim=0.126 | Sejong the Great: Sejong (Korean: 세종; Hanja: 世宗; May 15, 1397 – April 8, 1450), commonly known as Sejong the Great (세종대왕; 世宗大王), was the fourth monarch of the Koreanic state Jose...
   • sim=0.108 | Sejong the Great: Multiple places in South Korea, including Sejong Street, Sejong–Pocheon Expressway, and Sejong City, South Korea's de facto administrative capital, are named af...

✅ Wikipedia retrieval OK


## 4. Claim formation, NLI verifier, HalluMeasure-style CoT judge

In [12]:
# ─────────────────────────────────────────────────────────────
# CLAIM FORMATION — (question, answer) -> ONE declarative claim
# Both classes go through exactly this same path (no expanded-vs-raw asymmetry).
# ─────────────────────────────────────────────────────────────
STATEMENT_PROMPT = """Turn the question and answer below into ONE self-contained declarative sentence that states the answer as a fact.

Rules:
- Use only information contained in the question and the answer. Do NOT add, correct or verify any fact.
- Keep the answer exactly as given, even if you believe it is wrong.
- Name entities explicitly (no pronouns such as "it" or "he").
- Write it as a plain fact. NEVER mention "the question" or "the answer", and never write phrases like "is the answer to".
- Output ONLY the sentence: no quotes, no preamble, no explanation.

Question: {question}
Answer: {answer}"""

_REASONING_MARKERS = ["the user wants", "let me", "i need to", "i should", "wait,", "hmm",
                      "looking at", "the question asks", "the answer is given",
                      "is the answer to", "the answer to the question", "the question about", "in response to the question"]


def make_statement(question: str, answer: str) -> Tuple[str, bool, bool]:
    """Returns (statement, claim_ok, answer_kept). claim_ok=False means the LLM output was unusable
    and a plain 'question + answer' fallback string was used instead."""
    raw = call_with_fallback(
        step="claimgen",
        messages=[{"role": "user", "content": STATEMENT_PROMPT.format(question=question, answer=answer)}],
        temperature=0, max_tokens=200
    )
    statement = ""
    if raw:
        lines = [l.strip().strip('"').strip() for l in raw.strip().split("\n") if l.strip()]
        if lines:
            statement = lines[0]
    ok = (10 <= len(statement) <= 400) and not any(m in statement.lower() for m in _REASONING_MARKERS)
    if not ok:
        statement = f"Question: {question} Answer: {answer}"

    ans = answer.strip().strip(".").lower()
    if len(ans.split()) <= 6 and ans not in ("yes", "no"):
        answer_kept = ans in statement.lower()
    else:
        answer_kept = True          # yes/no and long answers: no cheap check, assume kept
    return statement, ok, answer_kept


# ─────────────────────────────────────────────────────────────
# NLI VERIFIER (baseline) — best entailment over the selected evidence chunks
# ─────────────────────────────────────────────────────────────
def verify_nli(claim: str, chunks: List[Dict]) -> Dict:
    if not chunks:
        return {"score": 0.0, "pred_hallucinated": True}
    best = max(get_entailment_score(claim, c["text"]) for c in chunks)
    return {"score": round(float(best), 4), "pred_hallucinated": best < NLI_THRESHOLD}


# ─────────────────────────────────────────────────────────────
# HalluMeasure-STYLE CoT JUDGE (Akbar et al., EMNLP 2024)
#   * one classification call per claim, chain-of-thought (5 steps) + few-shot examples
#   * five classes: supported / absent / contradicted / partially supported / unevaluatable
#   * hallucination = any class other than SUPPORTED
#   * error types: ENTITY, TEMPORAL and OVERGENERALIZATION are the ones named in the paper's public
#     write-up (the paper defines 10); OTHER is our catch-all.
#
# TWO MODES
#   claim-only          : free-form text (the demo) — the claim stands alone, exactly as in HalluMeasure.
#   question-conditioned: HaluEval QA — the claim is a proposed ANSWER to a QUESTION. HaluEval labels an answer
#     "hallucinated" when it does not correctly answer the question, which is not the same as the sentence being
#     false in isolation (e.g. "Alex Karras was born in 1935" is a true sentence but can still be a wrong answer).
#     So the judge is shown the question, told to treat the question's own premises as given, and to verify only
#     the proposed answer. This is our adaptation of HalluMeasure to QA data.
# ─────────────────────────────────────────────────────────────
JUDGE_LABELS = ["SUPPORTED", "CONTRADICTED", "ABSENT", "PARTIALLY_SUPPORTED", "UNEVALUATABLE"]

JUDGE_PROMPT = """You are a meticulous fact-checking judge. Decide whether a CLAIM is faithful to the CONTEXT.
Use ONLY the context. Never use your own knowledge of the world, even if you are sure the claim is true or false.
{qa_rules}
Work through these five steps. Write each step as ONE short sentence.
STEP 1 - Restate what must be verified: key entities, relation, and any dates or numbers.
STEP 2 - Find the context sentences that mention those entities.
STEP 3 - Compare the claim with those sentences detail by detail (entity, relation, date, number).
STEP 4 - Choose the class:
  SUPPORTED: every detail that must be verified is stated in, or directly implied by, the context.
  CONTRADICTED: the context states something that conflicts with the claim.
  ABSENT: the context does not confirm or deny the claim (the information is simply not there).
  PARTIALLY_SUPPORTED: part of the claim is supported, but another part is contradicted or absent.
  UNEVALUATABLE: the claim is an opinion, too vague, or cannot be checked against any context.
STEP 5 - If the class is not SUPPORTED, name the error type: ENTITY (wrong person, place or thing), TEMPORAL (wrong date, time or order), OVERGENERALIZATION (broader than the context allows), or OTHER. If SUPPORTED write NONE.

Finish with exactly these two lines and nothing after them:
VERDICT: <SUPPORTED | CONTRADICTED | ABSENT | PARTIALLY_SUPPORTED | UNEVALUATABLE>
ERROR_TYPE: <ENTITY | TEMPORAL | OVERGENERALIZATION | OTHER | NONE>

=== EXAMPLE 1 ===
CONTEXT: The Eiffel Tower, completed in 1889, stands on the Champ de Mars in Paris.
CLAIM: The Eiffel Tower was completed in 1889.
STEP 1 - Claim: Eiffel Tower; completed; 1889.
STEP 2 - The context says it was completed in 1889.
STEP 3 - The date matches exactly.
STEP 4 - SUPPORTED.
STEP 5 - NONE.
VERDICT: SUPPORTED
ERROR_TYPE: NONE

=== EXAMPLE 2 ===
CONTEXT: The Eiffel Tower, completed in 1889, stands on the Champ de Mars in Paris.
CLAIM: The Eiffel Tower was completed in 1925.
STEP 1 - Claim: Eiffel Tower; completed; 1925.
STEP 2 - The context says it was completed in 1889.
STEP 3 - 1925 conflicts with 1889.
STEP 4 - CONTRADICTED.
STEP 5 - TEMPORAL, the completion date is wrong.
VERDICT: CONTRADICTED
ERROR_TYPE: TEMPORAL

=== EXAMPLE 3 ===
CONTEXT: Hamlet is a tragedy written by William Shakespeare around 1600.
CLAIM: Hamlet was written by Christopher Marlowe.
STEP 1 - Claim: Hamlet; written by; Christopher Marlowe.
STEP 2 - The context says Hamlet was written by William Shakespeare.
STEP 3 - The author differs.
STEP 4 - CONTRADICTED.
STEP 5 - ENTITY, the wrong person is named.
VERDICT: CONTRADICTED
ERROR_TYPE: ENTITY

=== EXAMPLE 4 ===
CONTEXT: The Eiffel Tower, completed in 1889, stands on the Champ de Mars in Paris.
CLAIM: The Eiffel Tower is 330 metres tall.
STEP 1 - Claim: Eiffel Tower; height; 330 metres.
STEP 2 - The context mentions completion date and location only.
STEP 3 - Nothing in the context states a height.
STEP 4 - ABSENT.
STEP 5 - OTHER, the information is not in the context.
VERDICT: ABSENT
ERROR_TYPE: OTHER

=== EXAMPLE 5 ===
CONTEXT: Marie Curie won the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911.
CLAIM: Marie Curie won Nobel Prizes in Physics and Chemistry, both in 1903.
STEP 1 - Claim: Marie Curie; Physics and Chemistry prizes; both 1903.
STEP 2 - The context gives 1903 for Physics and 1911 for Chemistry.
STEP 3 - The Physics part matches; the Chemistry date does not.
STEP 4 - PARTIALLY_SUPPORTED.
STEP 5 - TEMPORAL, one date is wrong.
VERDICT: PARTIALLY_SUPPORTED
ERROR_TYPE: TEMPORAL
{qa_examples}
=== NOW JUDGE THIS ONE ===
{question_block}CONTEXT:
{context}

CLAIM: {claim}
"""

QA_RULES = """
The CLAIM is a proposed ANSWER to a QUESTION, written as a sentence. The QUESTION is shown only so you can interpret the claim.
- Assume the QUESTION's own premises are true. Do NOT try to verify details that come only from the question.
- Verify the PROPOSED ANSWER: is it a correct answer to the question according to the context?
- If the answer is true according to the context but is vague or does not actually answer what the question asks, choose PARTIALLY_SUPPORTED.
"""

QA_EXAMPLES = """
=== EXAMPLE 6 (question mode) ===
QUESTION: Which prize did the director of the film Amber Coast receive in 1993?
PROPOSED ANSWER: the Silver Compass prize
CONTEXT:
Doria Kessel (born 1958 in Porto) directed the 1991 film 'Amber Coast' and received the Silver Compass prize in 1993.

CLAIM: The director of Amber Coast received the Silver Compass prize in 1993.
STEP 1 - Verify: the prize is the Silver Compass prize, received in 1993 by the director of Amber Coast.
STEP 2 - The context says Doria Kessel directed Amber Coast and received the Silver Compass prize in 1993.
STEP 3 - The proposed answer matches; the film and year come from the question and are consistent anyway.
STEP 4 - SUPPORTED.
STEP 5 - NONE.
VERDICT: SUPPORTED
ERROR_TYPE: NONE

=== EXAMPLE 7 (question mode) ===
QUESTION: Which prize did the director of the film Amber Coast receive in 1993?
PROPOSED ANSWER: the Golden Anchor prize
CONTEXT:
Doria Kessel (born 1958 in Porto) directed the 1991 film 'Amber Coast' and received the Silver Compass prize in 1993.

CLAIM: The director of Amber Coast received the Golden Anchor prize in 1993.
STEP 1 - Verify: the prize is the Golden Anchor prize.
STEP 2 - The context says the 1993 prize was the Silver Compass prize.
STEP 3 - The named prize differs.
STEP 4 - CONTRADICTED.
STEP 5 - ENTITY, the wrong prize is named.
VERDICT: CONTRADICTED
ERROR_TYPE: ENTITY

=== EXAMPLE 8 (question mode) ===
QUESTION: Which prize did the director of the film Amber Coast receive in 1993?
PROPOSED ANSWER: Doria Kessel was born in Porto.
CONTEXT:
Doria Kessel (born 1958 in Porto) directed the 1991 film 'Amber Coast' and received the Silver Compass prize in 1993.

CLAIM: Doria Kessel, the director of Amber Coast, was born in Porto.
STEP 1 - Verify: the director was born in Porto.
STEP 2 - The context says Doria Kessel was born in 1958 in Porto.
STEP 3 - The statement is true, but the question asks which prize she received and this does not say.
STEP 4 - PARTIALLY_SUPPORTED.
STEP 5 - OTHER, true but it does not answer the question.
VERDICT: PARTIALLY_SUPPORTED
ERROR_TYPE: OTHER
"""

_VERDICT_RE = re.compile(r"VERDICT\s*:\s*[*_`\s]*(SUPPORTED|CONTRADICTED|ABSENT|PARTIALLY[_\s-]?SUPPORTED|UNEVALUATABLE)", re.I)
_ERRTYPE_RE = re.compile(r"ERROR[_\s]?TYPE\s*:\s*[*_`\s]*(ENTITY|TEMPORAL|OVERGENERALI[SZ]ATION|OTHER|NONE)", re.I)


def parse_judge_output(text: Optional[str]) -> Tuple[Optional[str], Optional[str]]:
    """Return (label, error_type); label is None when no valid VERDICT line exists.
    The LAST verdict wins, so any reasoning that leaks 'VERDICT' earlier is ignored."""
    if not text:
        return None, None
    verdicts = _VERDICT_RE.findall(text)
    if not verdicts:
        return None, None
    label = verdicts[-1].upper().replace(" ", "_").replace("-", "_")
    if label.startswith("PARTIALLY"):
        label = "PARTIALLY_SUPPORTED"
    errs = _ERRTYPE_RE.findall(text)
    err  = errs[-1].upper().replace("OVERGENERALISATION", "OVERGENERALIZATION") if errs else None
    return label, err


def format_context(chunks: List[Dict]) -> str:
    return "\n".join(f"[{i}] {c['text']}" for i, c in enumerate(chunks, 1))


def build_judge_prompt(claim: str, chunks: List[Dict], question: Optional[str] = None,
                       answer: Optional[str] = None) -> str:
    """question given -> question-conditioned mode; question=None -> claim-only mode."""
    qa = question is not None
    qblock = (f"QUESTION (shown only to interpret the claim): {question}\n"
              f"PROPOSED ANSWER (verbatim; this is what must be verified): {answer if answer is not None else claim}\n"
              if qa else "")
    return JUDGE_PROMPT.format(qa_rules=QA_RULES if qa else "", qa_examples=QA_EXAMPLES if qa else "",
                               question_block=qblock, context=format_context(chunks), claim=claim)


def judge_claim_cot(claim: str, chunks: List[Dict], question: Optional[str] = None,
                    answer: Optional[str] = None) -> Dict:
    """One CoT classification call for ONE claim against the given evidence chunks."""
    if not chunks:
        # Nothing retrieved -> by definition the claim is absent from the context. No LLM call.
        return {"label": "ABSENT", "error_type": "OTHER", "reasoning": "No evidence was retrieved.",
                "parse_ok": True, "no_evidence": True}

    prompt = build_judge_prompt(claim, chunks, question, answer)
    messages = [{"role": "user", "content": prompt}]
    raw = None
    for attempt in range(2):
        raw = call_with_fallback(step="judge", messages=messages, temperature=0, max_tokens=JUDGE_MAX_TOKENS)
        label, err = parse_judge_output(raw)
        if label is not None:
            return {"label": label, "error_type": err, "reasoning": (raw or "")[:1500],
                    "parse_ok": True, "no_evidence": False}
        # retry once with a stricter reminder
        messages = [{"role": "user", "content": prompt +
                     "\nIMPORTANT: keep every step to one short sentence and END with the two required lines "
                     "(VERDICT and ERROR_TYPE)."}]
    return {"label": None, "error_type": None, "reasoning": (raw or "")[:1500],
            "parse_ok": False, "no_evidence": False}


print("✅ Claim formation + NLI verifier + HalluMeasure-style CoT judge (claim-only and question-conditioned modes) ready")


✅ Claim formation + NLI verifier + HalluMeasure-style CoT judge (claim-only and question-conditioned modes) ready


In [13]:
# ─────────────────────────────────────────────────────────────
# SELF-TESTS (no API calls) — parsing, chunking, ranking and prompt building behave as intended
# ─────────────────────────────────────────────────────────────
assert parse_judge_output("STEP 4 - x\nVERDICT: SUPPORTED\nERROR_TYPE: NONE") == ("SUPPORTED", "NONE")
assert parse_judge_output("VERDICT: **CONTRADICTED**\nERROR_TYPE: Temporal")   == ("CONTRADICTED", "TEMPORAL")
assert parse_judge_output("VERDICT: partially supported\nERROR_TYPE: ENTITY")  == ("PARTIALLY_SUPPORTED", "ENTITY")
assert parse_judge_output("VERDICT: PARTIALLY_SUPPORTED")[0]                   == "PARTIALLY_SUPPORTED"
assert parse_judge_output("VERDICT: ABSENT\nblah\nVERDICT: SUPPORTED")[0]      == "SUPPORTED"   # last wins
assert parse_judge_output("I think it is fine.") == (None, None)
assert parse_judge_output(None) == (None, None)
assert parse_judge_output("VERDICT: NOT SUPPORTED")[0] is None                                   # ambiguous -> retry path

_doc = {"title": "T", "url": "u", "source": "test", "text":
        "== History ==\nThe city was founded in 1200. It grew slowly.\nThe university opened in 1850 and hosts 30000 students.\n"
        "Famous rivers flow nearby. The main airport handles 5 million passengers."}
_ev = select_evidence("When did the university open?", [_doc], top_k=2)
assert len(_ev) >= 1 and "1850" in _ev[0]["text"], _ev            # best chunk must contain the answer
assert chunk_text("== Heading ==\n") == []
assert select_evidence("anything", []) == []
assert all(len(c) <= CHUNK_CHARS + 60 for c in chunk_text("word " * 2000))                       # hard-split works

# v3.1: a short heading-like chunk that repeats the query words must NOT outrank the real sentence
_head = {"title": "H", "url": "h", "source": "test", "text": "Sejong Institute Foundation"}
_real = {"title": "R", "url": "r", "source": "test", "text":
         "Sejong the Great was the fourth monarch of Joseon and ruled the kingdom from 1418 until his death in 1450, "
         "a reign remembered for advances in science and the creation of the Korean alphabet."}
_ev2 = select_evidence("Sejong Institute Foundation Sejong ruled 1418", [_head, _real], top_k=2)
assert _ev2 and _ev2[0]["title"] == "R", _ev2

# v3.1: judge prompts — both modes fully formatted, question only present in QA mode
_ch = [{"text": "Some context sentence about Doria and a film that is long enough to count as evidence."}]
_p_qa = build_judge_prompt("Doria won the prize.", _ch, question="Who won the prize?", answer="Doria")
_p_co = build_judge_prompt("Doria won the prize.", _ch)
for _p in (_p_qa, _p_co):
    assert "{" not in _p and "}" not in _p, "unfilled placeholder in judge prompt"
assert "QUESTION (shown only" in _p_qa and "PROPOSED ANSWER (verbatim" in _p_qa and "EXAMPLE 6" in _p_qa
assert "QUESTION" not in _p_co and "PROPOSED ANSWER" not in _p_co and "EXAMPLE 6" not in _p_co
assert _p_qa.rstrip().endswith("CLAIM: Doria won the prize.") and _p_co.rstrip().endswith("CLAIM: Doria won the prize.")
print("✅ Self-tests passed (parser, chunker, ranker, short-chunk filter, judge prompts)")


✅ Self-tests passed (parser, chunker, ranker, short-chunk filter, judge prompts)


In [14]:
# LIVE JUDGE SMOKE TEST (6 API calls) — confirms the model follows the format and both modes behave.
# Context, questions and claims are invented here, so nothing can leak from HaluEval or from the few-shot examples.
_ctx = [{"text": "Aurelia Vane (born 1961 in Lisbon) directed the 1994 film 'Silver Harbor' and won the Golden Lantern award in 1996.",
         "title": "test", "url": "test", "source": "test"}]

print("── claim-only mode ──")
_cases = [
    ("Aurelia Vane directed the film Silver Harbor.",            {"SUPPORTED"}),
    ("Aurelia Vane won the Golden Lantern award in 2003.",       {"CONTRADICTED"}),
    ("Aurelia Vane is married to a famous architect.",           {"ABSENT"}),
]
_ok = 0
for _claim, _expected in _cases:
    _j = judge_claim_cot(_claim, _ctx)
    _hit = _j["label"] in _expected
    _ok += _hit
    print(f"{'✅' if _hit else '⚠️ '} expected {'/'.join(sorted(_expected)):<22} got {str(_j['label']):<20} parse_ok={_j['parse_ok']}   | {_claim}")

print("\n── question-conditioned mode (HaluEval QA) ──")
_q = "Which award did the director of the film Silver Harbor win in 1996?"
_qa_cases = [
    ("the Golden Lantern award", "The director of Silver Harbor won the Golden Lantern award in 1996.", {"SUPPORTED"}),
    ("the Crystal Compass award", "The director of Silver Harbor won the Crystal Compass award in 1996.", {"CONTRADICTED"}),
    ("Aurelia Vane was born in Lisbon.", "Aurelia Vane, the director of Silver Harbor, was born in Lisbon.",
        {"PARTIALLY_SUPPORTED", "UNEVALUATABLE", "ABSENT"}),          # true but does not answer -> must NOT be SUPPORTED
]
_okq = 0
for _ans, _claim, _expected in _qa_cases:
    _j = judge_claim_cot(_claim, _ctx, question=_q, answer=_ans)
    _hit = _j["label"] in _expected
    _okq += _hit
    print(f"{'✅' if _hit else '⚠️ '} expected {'/'.join(sorted(_expected)):<40} got {str(_j['label']):<20} parse_ok={_j['parse_ok']}   | answer: {_ans}")

print(f"\nclaim-only {_ok}/3, question-conditioned {_okq}/3.")
if _ok + _okq < 6:
    print("If a case is off, that is one LLM sample — but if parse_ok is False the model is ignoring the format. Last reasoning:\n", _j["reasoning"][:800])


── claim-only mode ──
✅ expected SUPPORTED              got SUPPORTED            parse_ok=True   | Aurelia Vane directed the film Silver Harbor.
✅ expected CONTRADICTED           got CONTRADICTED         parse_ok=True   | Aurelia Vane won the Golden Lantern award in 2003.
✅ expected ABSENT                 got ABSENT               parse_ok=True   | Aurelia Vane is married to a famous architect.

── question-conditioned mode (HaluEval QA) ──
✅ expected SUPPORTED                                got SUPPORTED            parse_ok=True   | answer: the Golden Lantern award
✅ expected CONTRADICTED                             got CONTRADICTED         parse_ok=True   | answer: the Crystal Compass award
✅ expected ABSENT/PARTIALLY_SUPPORTED/UNEVALUATABLE got PARTIALLY_SUPPORTED  parse_ok=True   | answer: Aurelia Vane was born in Lisbon.

claim-only 3/3, question-conditioned 3/3.


## 5. Judge-gated two-tier pipeline + resumable experiment runner

In [15]:
# ─────────────────────────────────────────────────────────────
# TWO-TIER VERIFICATION
#   Tier 1: Wikipedia (full pages, chunked + ranked). The CoT judge reads the evidence; if it is CONCLUSIVE
#           (SUPPORTED or CONTRADICTED) we stop.
#   Tier 2: only if Tier 1 is inconclusive (ABSENT / PARTIALLY_SUPPORTED / UNEVALUATABLE / unparsed):
#           add Firecrawl web pages to the evidence pool and judge again.
# (v3.0 escalated whenever NLI entailment < 0.5, which happened for 92% of claims.)
# Retrieval results are cached on disk, and the NLI baseline is scored on the same final evidence.
# ─────────────────────────────────────────────────────────────
RETRIEVAL_CACHE_PATH = os.path.join(WORK_DIR, "retrieval_cache_v31.json")
try:
    with open(RETRIEVAL_CACHE_PATH) as _f:
        RETRIEVAL_CACHE = json.load(_f)
except Exception:
    RETRIEVAL_CACHE = {}


def _save_retrieval_cache():
    tmp = RETRIEVAL_CACHE_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(RETRIEVAL_CACHE, f)
    os.replace(tmp, RETRIEVAL_CACHE_PATH)


def _cache_key(kind: str, question: str, statement: str) -> str:
    cfg = f"{CHUNK_CHARS}|{MIN_CHUNK_CHARS}|{TOP_K_CHUNKS}|{EVIDENCE_CHAR_BUDGET}"
    return hashlib.sha1(f"{kind}|{cfg}||{question}||{statement}".encode("utf-8")).hexdigest()


def wiki_evidence(question: str, statement: str) -> Dict:
    key = _cache_key("wiki", question, statement)
    if key in RETRIEVAL_CACHE:
        return RETRIEVAL_CACHE[key]
    docs = wikipedia_docs(statement, question)
    out = {"chunks": select_evidence(f"{statement} {question}", docs), "n_wiki_docs": len(docs)}
    if docs:                                    # never cache an empty result (may be an API hiccup)
        RETRIEVAL_CACHE[key] = out
        _save_retrieval_cache()
    return out


def firecrawl_evidence(question: str, statement: str, wiki_chunks: List[Dict]) -> Dict:
    key = _cache_key("fc", question, statement)
    if key in RETRIEVAL_CACHE:
        return RETRIEVAL_CACHE[key]
    web_docs = firecrawl_docs(statement)
    time.sleep(FIRECRAWL_SLEEP)
    pooled = [{"title": c["title"], "url": c["url"], "text": c["text"], "source": c["source"]} for c in wiki_chunks]
    out = {"chunks": select_evidence(f"{statement} {question}", pooled + web_docs), "n_web_docs": len(web_docs)}
    if web_docs:
        RETRIEVAL_CACHE[key] = out
        _save_retrieval_cache()
    return out


def verify_two_tier(question: str, statement: str, answer: Optional[str] = None,
                    use_question_in_judge: bool = True) -> Dict:
    """Returns {tier, chunks, judgment, tier1_label, n_wiki_docs, n_web_docs}."""
    jq = question if use_question_in_judge else None
    ja = answer   if use_question_in_judge else None

    wiki = wiki_evidence(question, statement)
    j1   = judge_claim_cot(statement, wiki["chunks"], question=jq, answer=ja)
    base = {"tier1_label": j1["label"], "n_wiki_docs": wiki["n_wiki_docs"]}

    if j1["label"] in CONCLUSIVE_LABELS:                                   # Tier 1 is enough
        return {**base, "tier": "wikipedia", "chunks": wiki["chunks"], "judgment": j1, "n_web_docs": 0}

    fc = firecrawl_evidence(question, statement, wiki["chunks"])           # Tier 2
    if fc["n_web_docs"] == 0:                                              # web gave nothing new: same evidence, reuse verdict
        return {**base, "tier": "firecrawl", "chunks": wiki["chunks"], "judgment": j1, "n_web_docs": 0}
    j2 = judge_claim_cot(statement, fc["chunks"], question=jq, answer=ja)
    return {**base, "tier": "firecrawl", "chunks": fc["chunks"], "judgment": j2, "n_web_docs": fc["n_web_docs"]}


def _pack_evidence(chunks: List[Dict]) -> List[Dict]:
    return [{"source": c["source"], "url": c["url"], "sim": c.get("sim"), "text": c["text"][:400]} for c in chunks]


# ─────────────────────────────────────────────────────────────
# ONE HaluEval SAMPLE  ->  verdicts for the ablation
#   web  = live retrieval (the full system)         orc = HaluEval gold knowledge (verifier ceiling)
#   NLI baseline and CoT judge both see the SAME evidence; orc_claimonly = v3.0 setting (question hidden)
# ─────────────────────────────────────────────────────────────
def process_sample(s: Dict) -> Dict:
    statement, claim_ok, answer_kept = make_statement(s["question"], s["answer"])

    web = verify_two_tier(s["question"], statement, answer=s["answer"], use_question_in_judge=True)
    orc = select_evidence(f"{statement} {s['question']}",
                          [{"title": "HaluEval knowledge", "url": "halueval:knowledge",
                            "text": s["knowledge"], "source": "oracle"}])

    web_nli = verify_nli(statement, web["chunks"])
    orc_nli = verify_nli(statement, orc)
    web_cot = web["judgment"]
    orc_cot = judge_claim_cot(statement, orc, question=s["question"], answer=s["answer"])   # question-conditioned
    orc_co  = judge_claim_cot(statement, orc)                                                # claim-only (v3.0 setting)

    return {
        "key": s["key"], "question": s["question"], "ground_truth": s["ground_truth"],
        "answer": s["answer"], "statement": statement, "claim_ok": claim_ok, "answer_kept": answer_kept,
        "tier": web["tier"], "tier1_label": web["tier1_label"],
        "n_wiki_docs": web["n_wiki_docs"], "n_web_docs": web["n_web_docs"],
        "n_web_chunks": len(web["chunks"]), "web_evidence": _pack_evidence(web["chunks"]),
        "web_nli_score": web_nli["score"],
        "web_cot_label": web_cot["label"], "web_cot_error_type": web_cot["error_type"],
        "web_cot_reasoning": web_cot["reasoning"], "web_cot_parse_ok": web_cot["parse_ok"],
        "orc_nli_score": orc_nli["score"],
        "orc_cot_label": orc_cot["label"], "orc_cot_error_type": orc_cot["error_type"],
        "orc_cot_reasoning": orc_cot["reasoning"], "orc_cot_parse_ok": orc_cot["parse_ok"],
        "orc_claimonly_label": orc_co["label"], "orc_claimonly_parse_ok": orc_co["parse_ok"],
    }


# ─────────────────────────────────────────────────────────────
# RESUMABLE RUNNER — the results file records its configuration, so old/mixed runs can never be resumed by accident
# ─────────────────────────────────────────────────────────────
def make_run_config() -> Dict:
    return {"version": "v3.1", "n_pairs": N_PAIRS, "seed": EVAL_SEED, "nli_threshold": NLI_THRESHOLD,
            "top_k": TOP_K_CHUNKS, "chunk_chars": CHUNK_CHARS, "min_chunk_chars": MIN_CHUNK_CHARS,
            "conclusive": list(CONCLUSIVE_LABELS),
            "judge_models": FALLBACK_MODELS["judge"], "claimgen_models": FALLBACK_MODELS["claimgen"]}


def load_state(path: str, config: Dict) -> Dict:
    if os.path.exists(path):
        try:
            with open(path) as f:
                state = json.load(f)
            if state.get("config") == config and isinstance(state.get("rows"), list):
                print(f"📂 Resuming: {len(state['rows'])} samples already done")
                return state
            print("⚠️  Results file was produced with a different configuration -> moved to .bak, starting fresh")
        except Exception as e:
            print(f"⚠️  Could not read results file ({e}) -> moved to .bak, starting fresh")
        try:
            os.replace(path, path + ".bak")
        except Exception:
            pass
    return {"config": config, "rows": []}


def save_state(path: str, state: Dict):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=1)
    os.replace(tmp, path)


def run_experiment(samples: List[Dict], save_path: str) -> Tuple[List[Dict], List[Tuple[str, str]]]:
    state     = load_state(save_path, make_run_config())
    done_keys = {r["key"] for r in state["rows"]}
    remaining = [s for s in samples if s["key"] not in done_keys]
    print(f"\n🔬 HaluEval v3.1 experiment | total={len(samples)}  done={len(state['rows'])}  remaining={len(remaining)}")

    failures: List[Tuple[str, str]] = []
    for s in tqdm(remaining, desc="HaluEval v3.1"):
        try:
            state["rows"].append(process_sample(s))
            save_state(save_path, state)                    # save after EVERY sample (safe to interrupt)
        except Exception as e:
            failures.append((s["question"][:80], repr(e)[:150]))
            print(f"\n   ⚠️  Sample failed (will retry on next run): {repr(e)[:150]}")
    return state["rows"], failures


print("✅ Two-tier verification (judge-gated), per-sample pipeline and resumable runner ready")


✅ Two-tier verification (judge-gated), per-sample pipeline and resumable runner ready


## 6. HaluEval sample (balanced: N truthful + N hallucinated, same questions)

In [16]:
HALUEVAL_PATHS = ["data/qa_data.json", "qa_data.json"]


def load_halueval(filepath: str) -> List[Dict]:
    """Accepts either JSON Lines (one object per line — the HaluEval format) or a single JSON array."""
    with open(filepath, encoding="utf-8") as f:
        raw = f.read().strip()
    if raw.startswith("["):
        return json.loads(raw)
    return [json.loads(line) for line in raw.splitlines() if line.strip()]


def find_halueval_path() -> Optional[str]:
    for p in HALUEVAL_PATHS:
        if os.path.exists(p):
            return p
    for root, _, files in os.walk("/kaggle/input"):          # fall back to auto-discovery
        for fn in files:
            if "qa_data" in fn.lower() and fn.lower().endswith((".json", ".jsonl")):
                return os.path.join(root, fn)
    return None


def build_samples(data: List[Dict], n_pairs: int, seed: int) -> List[Dict]:
    """n_pairs distinct questions -> 2*n_pairs samples (right answer = truthful, hallucinated answer = hallucinated).
    Uses a local RNG, so the draw depends ONLY on (data, n_pairs, seed)."""
    need = ("knowledge", "question", "right_answer", "hallucinated_answer")
    pool, seen_q = [], set()
    for d in data:
        if not all(isinstance(d.get(k), str) and d[k].strip() for k in need):
            continue
        if d["question"] in seen_q:
            continue
        seen_q.add(d["question"])
        pool.append(d)
    if len(pool) < n_pairs:
        raise ValueError(f"Only {len(pool)} valid questions available, need {n_pairs}")

    rng    = random.Random(seed)
    chosen = rng.sample(pool, n_pairs)
    samples = []
    for d in chosen:
        for gt, ans in (("truthful", d["right_answer"]), ("hallucinated", d["hallucinated_answer"])):
            samples.append({"question": d["question"], "knowledge": d["knowledge"], "answer": ans.strip(),
                            "ground_truth": gt, "key": f"{d['question']}||{gt}"})
    rng.shuffle(samples)
    return samples


_path = find_halueval_path()
assert _path, "HaluEval file not found — add your dataset to the notebook and update HALUEVAL_PATHS"
halueval_data = load_halueval(_path)
halueval_samples = build_samples(halueval_data, N_PAIRS, EVAL_SEED)

print(f"✅ Loaded {len(halueval_data)} HaluEval records from: {_path}")
print(f"✅ Built {len(halueval_samples)} samples = {N_PAIRS} truthful + {N_PAIRS} hallucinated (seed={EVAL_SEED})")
_wl = pd.Series([len(s["answer"].split()) for s in halueval_samples], index=[s["ground_truth"] for s in halueval_samples])
print("   Mean answer length (words) by class — the two classes are now processed identically regardless:")
print(_wl.groupby(level=0).mean().round(1).to_string())


✅ Loaded 10000 HaluEval records from: /kaggle/input/datasets/hemantsai/qadata/qa_data (2).json
✅ Built 100 samples = 50 truthful + 50 hallucinated (seed=11)
   Mean answer length (words) by class — the two classes are now processed identically regardless:
hallucinated    12.7
truthful         2.2


## 7. Run the experiment (resumable)

# RUN THE EXPERIMENT (resumable — if Kaggle stops, just run this cell again)
RESULTS_PATH = os.path.join(WORK_DIR, f"halueval_v31_seed{EVAL_SEED}_n{N_PAIRS}.json")
rows, failures = run_experiment(halueval_samples, RESULTS_PATH)

print(f"\n✅ Finished: {len(rows)}/{len(halueval_samples)} samples complete, {len(failures)} failed")
for q, err in failures:
    print(f"   ❌ {q}... -> {err}")
print(f"   Wikipedia API : {WIKI_STATS['requests']} requests, {WIKI_STATS['retried']} retried, {WIKI_STATS['errors']} failed for good")
print(f"   Firecrawl     : {FIRECRAWL_STATS['searches']} searches, {FIRECRAWL_STATS['retried']} retried, {FIRECRAWL_STATS['errors']} failed for good"
      + (f"  (last error: {FIRECRAWL_STATS['last_error']})" if FIRECRAWL_STATS['errors'] else ""))


In [19]:
# RUN THE EXPERIMENT (resumable — if Kaggle stops, just run this cell again)
RESULTS_PATH = os.path.join(WORK_DIR, f"halueval_v31_seed{EVAL_SEED}_n{N_PAIRS}.json")
rows, failures = run_experiment(halueval_samples, RESULTS_PATH)

print(f"\n✅ Finished: {len(rows)}/{len(halueval_samples)} samples complete, {len(failures)} failed")
for q, err in failures:
    print(f"   ❌ {q}... -> {err}")
print(f"   Wikipedia API : {WIKI_STATS['requests']} requests, {WIKI_STATS['retried']} retried, {WIKI_STATS['errors']} failed for good")
print(f"   Firecrawl     : {FIRECRAWL_STATS['searches']} searches, {FIRECRAWL_STATS['retried']} retried, {FIRECRAWL_STATS['errors']} failed for good"
      + (f"  (last error: {FIRECRAWL_STATS['last_error']})" if FIRECRAWL_STATS['errors'] else ""))


📂 Resuming: 90 samples already done

🔬 HaluEval v3.1 experiment | total=100  done=90  remaining=10


HaluEval v3.1:  20%|██        | 2/10 [00:43<02:52, 21.57s/it]

   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 20s...
   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 40s...
   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 20s...


HaluEval v3.1:  40%|████      | 4/10 [02:57<04:46, 47.73s/it]

   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 20s...
   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 40s...


HaluEval v3.1:  60%|██████    | 6/10 [05:03<03:28, 52.06s/it]

... (3 outputs left out)

HaluEval v3.1:  80%|████████  | 8/10 [06:59<01:50, 55.12s/it]

   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 20s...


HaluEval v3.1: 100%|██████████| 10/10 [08:24<00:00, 50.45s/it]


✅ Finished: 100/100 samples complete, 0 failed
   Wikipedia API : 454 requests, 0 retried, 0 failed for good
   Firecrawl     : 69 searches, 0 retried, 0 failed for good


## 8. Results — ablation, paired tests, tier usage, pipeline health

In [20]:
from sklearn.metrics import roc_auc_score


def _show(x):
    try:
        display(x)
    except NameError:
        print(x.to_string() if hasattr(x, "to_string") else x)


def wilson_ci(k: int, n: int, z: float = 1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    denom  = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half   = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return float(centre - half), float(centre + half)


def _lab_to_pred(l, rule):
    if l is None or (isinstance(l, float) and np.isnan(l)):
        return None
    if rule == "strict":                 # HalluMeasure: hallucination = anything other than SUPPORTED
        return "truthful" if l == "SUPPORTED" else "hallucinated"
    return "hallucinated" if l in ("CONTRADICTED", "ABSENT") else "truthful"     # lenient


def build_results_df(rows: List[Dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for pre in ("web", "orc"):
        df[f"{pre}_nli_pred"]          = np.where(df[f"{pre}_nli_score"] < NLI_THRESHOLD, "hallucinated", "truthful")
        df[f"{pre}_cot_pred"]          = df[f"{pre}_cot_label"].map(lambda l: _lab_to_pred(l, "strict"))
        df[f"{pre}_cot_pred_lenient"]  = df[f"{pre}_cot_label"].map(lambda l: _lab_to_pred(l, "lenient"))
    df["orc_claimonly_pred"] = df["orc_claimonly_label"].map(lambda l: _lab_to_pred(l, "strict"))
    return df


def condition_metrics(df: pd.DataFrame, name: str, pred_col: str, score_col: Optional[str] = None) -> Dict:
    d = df[df[pred_col].notna()]
    n = len(d)
    out = {"condition": name, "n": n, "unparsed": len(df) - n}
    if n == 0:
        return out
    y = (d["ground_truth"] == "hallucinated").astype(int).values
    p = (d[pred_col] == "hallucinated").astype(int).values
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    acc  = (tp + tn) / n
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    lo, hi = wilson_ci(tp + tn, n)
    auc = np.nan
    if score_col is not None and len(set(y)) == 2:
        auc = float(roc_auc_score(y, 1 - d[score_col].values))      # low entailment = more likely hallucinated
    g = d.assign(correct=(d[pred_col] == d["ground_truth"])).groupby("question")["correct"].agg(["count", "all"])
    g = g[g["count"] == 2]
    pair_acc = float(g["all"].mean()) if len(g) else np.nan
    out.update({"accuracy": acc, "ci95_low": lo, "ci95_high": hi, "precision": prec, "recall": rec, "f1": f1,
                "auc": auc, "pair_acc": pair_acc, "TP": tp, "FP": fp, "FN": fn, "TN": tn})
    return out


results_df = build_results_df(rows)

conditions = [
    ("1. Web + NLI            (baseline)",             "web_nli_pred",          "web_nli_score"),
    ("2. Web + CoT judge      (WebHaluCheck)",         "web_cot_pred",          None),
    ("3. Oracle + NLI         (NLI ceiling)",          "orc_nli_pred",          "orc_nli_score"),
    ("4. Oracle + CoT judge   (HalluMeasure setting)", "orc_cot_pred",         None),
    ("4c. Oracle + CoT, claim-only (v3.0 setting)",    "orc_claimonly_pred",    None),
    ("2b. Web + CoT, lenient rule",                    "web_cot_pred_lenient",  None),
    ("4b. Oracle + CoT, lenient rule",                 "orc_cot_pred_lenient",  None),
]
summary = pd.DataFrame([condition_metrics(results_df, *c) for c in conditions]).set_index("condition")

print("═" * 78)
print(f"  RESULTS — HaluEval QA, {len(results_df)} samples ({results_df['question'].nunique()} question pairs), seed={EVAL_SEED}")
print("  Positive class = 'hallucinated'. Chance level = 0.50.")
print("═" * 78)
_show(summary.round(3))
print("\nNotes: 'strict' = hallucination if the judge says anything other than SUPPORTED (HalluMeasure's rule);")
print("       'lenient' = only CONTRADICTED/ABSENT count. pair_acc = both answers to the SAME question judged correctly.")
print("       auc is only defined for NLI (it has a continuous score). ci95 = Wilson 95% interval on accuracy.")

# ── Paired significance (exact McNemar): is one condition really better than another on the SAME samples? ──
from scipy.stats import binomtest


def paired_test(df: pd.DataFrame, pred_a: str, pred_b: str) -> Dict:
    d  = df[df[pred_a].notna() & df[pred_b].notna()]
    ca = (d[pred_a] == d["ground_truth"]).values
    cb = (d[pred_b] == d["ground_truth"]).values
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())          # A right/B wrong ; A wrong/B right
    p = float(binomtest(min(b, c), b + c, 0.5).pvalue) if (b + c) else 1.0
    return {"acc_A": ca.mean(), "acc_B": cb.mean(), "diff (A-B)": ca.mean() - cb.mean(),
            "A_only_right": b, "B_only_right": c, "p_value": p}


_cmp = [
    ("Web CoT       vs Web NLI          (is the judge better than NLI?)",        "web_cot_pred",       "web_nli_pred"),
    ("Oracle CoT    vs Oracle NLI       (same, with perfect evidence)",          "orc_cot_pred",       "orc_nli_pred"),
    ("Oracle CoT(q) vs Oracle CoT(claim-only)  (effect of using the question)", "orc_cot_pred",       "orc_claimonly_pred"),
    ("Oracle CoT    vs Web CoT          (cost of imperfect retrieval)",          "orc_cot_pred",       "web_cot_pred"),
]
_pt = pd.DataFrame([{"comparison": n, **paired_test(results_df, a, b)} for n, a, b in _cmp]).set_index("comparison")
print("\n📐 PAIRED TESTS (exact McNemar; p < 0.05 = difference unlikely to be chance)")
_show(_pt.round(3))
print("Do not claim an improvement unless p < 0.05. A large gap with p > 0.05 means 'suggestive, needs more samples'.")

# ── Tier usage (the credit-savings claim) ────────────────────────────────────
print("\n📈 TIER USAGE — which tier produced the evidence")
print((results_df["tier"].value_counts(normalize=True) * 100).round(1).to_string())
print(pd.crosstab(results_df["ground_truth"], results_df["tier"]).to_string())
print("Tier 1 (Wikipedia) is accepted when the judge is conclusive (SUPPORTED/CONTRADICTED); otherwise Firecrawl is used.")
print("Tier-1 verdicts before escalation:")
print(results_df["tier1_label"].fillna("PARSE_FAIL").value_counts().to_string())

# ── Pipeline health (rules out silent failures) ──────────────────────────────
print("\n🩺 PIPELINE HEALTH")
print(f"   Wikipedia API (this session) : {WIKI_STATS['requests']} requests, {WIKI_STATS['retried']} retried, {WIKI_STATS['errors']} failed for good")
print(f"   Samples with NO Wikipedia page: {(results_df['n_wiki_docs'] == 0).sum()}")
print(f"   Escalated but Firecrawl gave 0 usable pages: {((results_df['tier'] == 'firecrawl') & (results_df['n_web_docs'] == 0)).sum()}  (if large: check Firecrawl credits/key)")
print(f"   Claim formation OK           : {results_df['claim_ok'].mean() * 100:.0f}%   answer preserved in claim: {results_df['answer_kept'].mean() * 100:.0f}%")
print(f"   Samples with NO web evidence : {(results_df['n_web_chunks'] == 0).sum()}")
print(f"   Judge parse failures         : web={(~results_df['web_cot_parse_ok']).sum()}  oracle={(~results_df['orc_cot_parse_ok']).sum()}  oracle-claim-only={(~results_df['orc_claimonly_parse_ok']).sum()}")

print("\n🏷️  JUDGE LABELS vs ground truth (web evidence)")
print(pd.crosstab(results_df["ground_truth"], results_df["web_cot_label"].fillna("PARSE_FAIL")).to_string())
print("\n🏷️  JUDGE LABELS vs ground truth (oracle evidence)")
print(pd.crosstab(results_df["ground_truth"], results_df["orc_cot_label"].fillna("PARSE_FAIL")).to_string())

print("\n🏷️  JUDGE LABELS vs ground truth (oracle evidence, claim-only / question hidden — the v3.0 setting)")
print(pd.crosstab(results_df["ground_truth"], results_df["orc_claimonly_label"].fillna("PARSE_FAIL")).to_string())

print("\n🧩 ERROR TYPES among claims the web judge flagged (HalluMeasure-style breakdown)")
_flag = results_df[results_df["web_cot_pred"] == "hallucinated"]
print(_flag["web_cot_error_type"].fillna("UNKNOWN").value_counts().to_string() if len(_flag) else "   (none flagged)")

summary.to_csv(os.path.join(WORK_DIR, "v31_summary.csv"))
_pt.to_csv(os.path.join(WORK_DIR, "v31_paired_tests.csv"))
results_df.drop(columns=["web_evidence"]).to_csv(os.path.join(WORK_DIR, "v31_rows.csv"), index=False)
print(f"\n💾 Saved v31_summary.csv, v31_paired_tests.csv and v31_rows.csv to {WORK_DIR}")


══════════════════════════════════════════════════════════════════════════════
  RESULTS — HaluEval QA, 100 samples (50 question pairs), seed=11
  Positive class = 'hallucinated'. Chance level = 0.50.
══════════════════════════════════════════════════════════════════════════════


,n,unparsed,accuracy,ci95_low,ci95_high,precision,recall,f1,auc,pair_acc,TP,FP,FN,TN
condition,,,,,,,,,,,,,,
1. Web + NLI (baseline),100,0,0.60,0.502,0.691,0.562,0.90,0.692,0.619,0.26,45,35,5,15
2. Web + CoT judge (WebHaluCheck),100,0,0.67,0.573,0.754,0.613,0.92,0.736,NaN,0.36,46,29,4,21
3. Oracle + NLI (NLI ceiling),100,0,0.60,0.502,0.691,0.576,0.76,0.655,0.679,0.28,38,28,12,22
4. Oracle + CoT judge (HalluMeasure setting),100,0,0.79,0.700,0.858,0.774,0.82,0.796,NaN,0.60,41,12,9,38
"4c. Oracle + CoT, claim-only (v3.0 setting)",100,0,0.72,0.625,0.799,0.667,0.88,0.759,NaN,0.46,44,22,6,28
"2b. Web + CoT, lenient rule",100,0,0.69,0.594,0.772,0.673,0.74,0.705,NaN,0.46,37,18,13,32
"4b. Oracle + CoT, lenient rule",100,0,0.76,0.668,0.833,0.882,0.60,0.714,NaN,0.54,30,4,20,46



Notes: 'strict' = hallucination if the judge says anything other than SUPPORTED (HalluMeasure's rule);
       'lenient' = only CONTRADICTED/ABSENT count. pair_acc = both answers to the SAME question judged correctly.
       auc is only defined for NLI (it has a continuous score). ci95 = Wilson 95% interval on accuracy.

📐 PAIRED TESTS (exact McNemar; p < 0.05 = difference unlikely to be chance)


,acc_A,acc_B,diff (A-B),A_only_right,B_only_right,p_value
comparison,,,,,,
Web CoT vs Web NLI (is the judge better than NLI?),0.67,0.60,0.07,15,8,0.210
"Oracle CoT vs Oracle NLI (same, with perfect evidence)",0.79,0.60,0.19,25,6,0.001
Oracle CoT(q) vs Oracle CoT(claim-only) (effect of using the question),0.79,0.72,0.07,10,3,0.092
Oracle CoT vs Web CoT (cost of imperfect retrieval),0.79,0.67,0.12,20,8,0.036


Do not claim an improvement unless p < 0.05. A large gap with p > 0.05 means 'suggestive, needs more samples'.

📈 TIER USAGE — which tier produced the evidence
tier
firecrawl    70.0
wikipedia    30.0
tier          firecrawl  wikipedia
ground_truth                      
hallucinated         37         13
truthful             33         17
Tier 1 (Wikipedia) is accepted when the judge is conclusive (SUPPORTED/CONTRADICTED); otherwise Firecrawl is used.
Tier-1 verdicts before escalation:
tier1_label
ABSENT                 58
SUPPORTED              17
CONTRADICTED           13
PARTIALLY_SUPPORTED    12

🩺 PIPELINE HEALTH
   Wikipedia API (this session) : 454 requests, 0 retried, 0 failed for good
   Samples with NO Wikipedia page: 3
   Escalated but Firecrawl gave 0 usable pages: 3  (if large: check Firecrawl credits/key)
   Claim formation OK           : 100%   answer preserved in claim: 98%
   Samples with NO web evidence : 0
   Judge parse failures         : web=0  oracle=0  oracle-cla

## 9. Error analysis (use these examples in the viva)

In [21]:
def show_errors(df: pd.DataFrame, prefix: str = "web", kind: str = "false_positive", k: int = 5):
    """Print misclassified cases so you can explain the failures to the panel.
    kind='false_positive': truthful answer flagged as hallucinated.  kind='false_negative': hallucination missed."""
    pred = f"{prefix}_cot_pred"
    if kind == "false_positive":
        d = df[(df["ground_truth"] == "truthful") & (df[pred] == "hallucinated")]
    else:
        d = df[(df["ground_truth"] == "hallucinated") & (df[pred] == "truthful")]
    print(f"\n{'═' * 70}\n{prefix.upper()} / {kind}: {len(d)} cases (showing up to {k})\n{'═' * 70}")
    for _, r in d.head(k).iterrows():
        print(f"\n• CLAIM   : {r['statement']}")
        print(f"  GT={r['ground_truth']}  judge={r[f'{prefix}_cot_label']}  error_type={r[f'{prefix}_cot_error_type']}")
        print(f"  REASONING: {str(r[f'{prefix}_cot_reasoning'])[:350].replace(chr(10), ' ')}")
        if prefix == "web" and isinstance(r.get("web_evidence"), list) and r["web_evidence"]:
            e = r["web_evidence"][0]
            print(f"  TOP EVIDENCE ({e['source']}, {e['url']}): {e['text'][:220]}")


# Compare where the web judge goes wrong versus the oracle judge:
show_errors(results_df, "web", "false_positive", k=5)   # true answers flagged  -> usually a retrieval miss
show_errors(results_df, "web", "false_negative", k=5)   # hallucinations missed -> usually misleading evidence
show_errors(results_df, "orc", "false_positive", k=3)   # if these are many, the claim wording/judge is the issue
show_errors(results_df, "orc", "false_negative", k=3)



══════════════════════════════════════════════════════════════════════
WEB / false_positive: 29 cases (showing up to 5)
══════════════════════════════════════════════════════════════════════

• CLAIM   : Fashion Show Mall is the name of the business connected to the Treasure Island Hotel and Casino that is owned by General Growth Properties Inc.
  GT=truthful  judge=ABSENT  error_type=OTHER
  REASONING: STEP 1 - Verify: the business connected to Treasure Island Hotel and Casino owned by General Growth Properties Inc is Fashion Show Mall. STEP 2 - The context mentions General Growth Properties owns Fashion Show Mall (sentences 1, 2, 4) and that Treasure Island was sold to Phil Ruffin (sentence 3), but does not state any connection between Fashion S
  TOP EVIDENCE (firecrawl, https://en.wikipedia.org/wiki/Fashion_Show_Las_Vegas): General Growth Properties became a subsidiary of Brookfield Properties in 2018, and the company was rebranded back to the GGP name in January 2026.[\[3\]](http

## 10. Free-form demo (King Sejong, from the base paper)

In [22]:
EXTRACTION_PROMPT = """You are a claim extraction assistant.

Given the following LLM response, extract all individual factual claims from the ENTIRE response at once.

Rules you MUST follow strictly:
1. Each claim must be completely self-contained
2. Replace ALL pronouns (he, she, it, they, his, her, its, their) with the actual noun they refer to
3. Each claim must contain exactly ONE verifiable fact
4. Each claim must be under 20 words
5. Skip opinions, vague statements, and unverifiable claims
6. Skip claims like "X is considered important" — not verifiable
7. Output ONLY a numbered list. No preamble, no explanation, no extra text.

Response to analyze:
{response}"""

def extract_claims(response_text: str) -> List[str]:
    raw = call_with_fallback(
        step="extraction",
        messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(response=response_text)}],
        temperature=0, max_tokens=512
    )
    if not raw:
        return []

    REASONING_MARKERS = [
        "the user wants", "let me", "i need to", "i should", "wait -", "wait,",
        "hmm", "the response:", "looking at", "analyzing", "the rule says"
    ]

    claims = []
    for line in raw.strip().split("\n"):
        line = line.strip()
        cleaned = re.sub(r'^[\d]+[.)\s]+', '', line).strip()
        cleaned = re.sub(r'^[-•*]\s*', '', cleaned).strip()

        if not cleaned or len(cleaned) <= 10:
            continue
        # Skip reasoning/meta-commentary artifacts
        if any(marker in cleaned.lower() for marker in REASONING_MARKERS):
            continue
        # Skip claims that are just the whole original response pasted back
        if len(cleaned) > 250:
            continue
        # Skip claims way over the "under 20 words" rule — sign of leaked reasoning
        if len(cleaned.split()) > 35:
            continue

        claims.append(cleaned)
    return claims


In [23]:
TEST_RESPONSE = """King Sejong, also known as Sejong the Great, was the fourth king of the
Joseon Dynasty in Korea. He ruled from 1418 to 1450 and is considered one of the greatest
kings in Korean history. King Sejong is best known for his promotion of science, technology,
and culture, as well as his creation of the Korean alphabet, Hangul. He is also credited
with expanding Korean territory and implementing various social and economic reforms during
his reign. King Sejong encouraged active trade with the Qing Dynasty in China."""


def run_webhalucheck_v3(question: str, response: str, verbose: bool = True) -> Dict:
    """Free-form pipeline: extract claims -> judge-gated two-tier retrieval -> HalluMeasure-style CoT judge.
    Claims stand alone here (claim-only mode), exactly as in HalluMeasure. The judge's chain-of-thought IS the
    explanation (no extra explanation call needed)."""
    claims = extract_claims(response)
    result = {"question": question, "response": response, "claims": claims, "claim_results": []}
    for i, claim in enumerate(claims, 1):
        if verbose:
            print(f"   [{i}/{len(claims)}] {claim}")
        v   = verify_two_tier(question, claim, use_question_in_judge=False)
        j   = v["judgment"]
        nli = verify_nli(claim, v["chunks"])
        result["claim_results"].append({
            "claim": claim, "tier": v["tier"], "nli_score": nli["score"], "label": j["label"],
            "error_type": j["error_type"], "explanation": j["reasoning"],
            "flagged": (j["label"] != "SUPPORTED") if j["label"] else None,
            "evidence_urls": list(dict.fromkeys(c["url"] for c in v["chunks"]))[:3],
        })
    return result


def display_results_v3(res: Dict):
    SEP = "═" * 70
    crs = res["claim_results"]
    flagged = sum(1 for c in crs if c["flagged"])
    print(f"\n{SEP}\n  WEBHALUCHECK v3 — Two-tier retrieval + HalluMeasure-style CoT judge\n{SEP}")
    print(f"❓ {res['question']}\n")
    print(f"📊 Claims: {len(crs)} | Flagged (not SUPPORTED): {flagged} | Hallucination score: {flagged / max(len(crs), 1):.1%}")
    for i, c in enumerate(crs, 1):
        icon = "🔴" if c["flagged"] else ("🟢" if c["flagged"] is False else "⚪")
        print(f"\n{icon} CLAIM {i} [{c['tier']}]: {c['claim']}")
        print(f"     Judge: {c['label']} (error type: {c['error_type']}) | NLI entailment: {c['nli_score']:.3f}")
        for u in c["evidence_urls"]:
            print(f"     source: {u}")
        if c["flagged"]:
            print(f"     💬 Chain-of-thought explanation:\n        " + c["explanation"].replace("\n", "\n        "))
    print(SEP)


print("🧪 Demo — King Sejong example from the base paper (the Qing Dynasty sentence is the planted hallucination)\n")
demo = run_webhalucheck_v3("Who was King Sejong?", TEST_RESPONSE)
display_results_v3(demo)


🧪 Demo — King Sejong example from the base paper (the Qing Dynasty sentence is the planted hallucination)

   [1/6] King Sejong was the fourth king of the Joseon Dynasty in Korea.
   [2/6] King Sejong ruled from 1418 to 1450.
   [3/6] King Sejong created the Korean alphabet Hangul.
   [4/6] King Sejong expanded Korean territory.
   [5/6] King Sejong implemented various social and economic reforms during his reign.
   [6/6] King Sejong encouraged active trade with the Qing Dynasty in China.
   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 20s...
   ⚠️  Rate limit hit on nvidia/nemotron-3-ultra-550b-a55b. Waiting 40s...

══════════════════════════════════════════════════════════════════════
  WEBHALUCHECK v3 — Two-tier retrieval + HalluMeasure-style CoT judge
══════════════════════════════════════════════════════════════════════
❓ Who was King Sejong?

📊 Claims: 6 | Flagged (not SUPPORTED): 2 | Hallucination score: 33.3%

🟢 CLAIM 1 [firecrawl]: King Sejong was the four

## Notes for reporting
* **Report the table exactly as produced.** Do not tune prompts or thresholds after seeing the numbers on this sample. If you want to iterate, change `EVAL_SEED` (fresh sample) or use a small dev run first, then freeze everything and do one final run.
* `Oracle` rows use HaluEval's own gold `knowledge` as evidence — they measure how good the *verifier* is when retrieval is perfect. `Web` rows measure the full system. The gap between them is the cost of retrieval.
* Row **4c** repeats the oracle judge with the question hidden (the v3.0 setting): the gap between 4 and 4c is the measured effect of question conditioning.
* `strict` vs `lenient` are two ways to turn the judge's 5 classes into a yes/no hallucination decision (see the analysis cell). HalluMeasure itself counts every non-SUPPORTED class as unsupported.
* Error types: the paper defines 10; this notebook uses the three named in its public write-up (ENTITY, TEMPORAL, OVERGENERALIZATION) plus OTHER.